<a href="https://colab.research.google.com/github/tukamilano/Semi-Thue/blob/main/Levenshtein_%E5%AD%A6%E7%BF%92%E5%89%8D%E5%87%A6%E7%90%86_%E5%90%88%E6%B3%95%E6%89%8B%E3%81%AE%E5%88%97%E6%8C%99.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Validな選択肢を二段階のdictを使って列挙

In [1]:
!pip install pyahocorasick

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.4 MB/s eta 0:00:00


In [2]:
import json

# アップロードしたファイルのパス
file_rule_path = "/content/rules.json"

# JSONを読み込む
with open(file_rule_path, "r") as f:
    data = json.load(f)

# 中身を確認
print(data.keys())       # どんなキーがあるか
print(len(data["rules"]))  # rulesの数を確認
print(data["rules"][:100])   # 最初の5個のルールを表示

dict_keys(['config', 'base_seed_rules', 'rules'])
500
[['', 'ABTUEIXIG'], ['', 'AGVKBDLTDIKCRX'], ['', 'B'], ['', 'BM'], ['', 'BQEIRC'], ['', 'CERURAK'], ['', 'CK'], ['', 'CQQS'], ['', 'CT'], ['', 'CVP'], ['', 'DYWCFXIKIKR'], ['', 'E'], ['', 'EEJJXT'], ['', 'EOYL'], ['', 'F'], ['', 'FGXBXPBCQFH'], ['', 'FMDCNG'], ['', 'FWESD'], ['', 'FYJRXXW'], ['', 'G'], ['', 'GA'], ['', 'GALIK'], ['', 'GNCUD'], ['', 'GNILMKAVOVJSK'], ['', 'GXT'], ['', 'H'], ['', 'HA'], ['', 'HUGMSWHOSXMNL'], ['', 'HYLF'], ['', 'I'], ['', 'ILOWXYMJ'], ['', 'IT'], ['', 'J'], ['', 'JNMDUBTESO'], ['', 'K'], ['', 'KKF'], ['', 'LKHMVYXY'], ['', 'LUHVO'], ['', 'M'], ['', 'MD'], ['', 'MHWVFMIKGV'], ['', 'MUPLTMQQ'], ['', 'N'], ['', 'NG'], ['', 'NLW'], ['', 'NLWXDHIUFR'], ['', 'OJA'], ['', 'OLPIKWAJTTMOONU'], ['', 'OOIQFA'], ['', 'OORGOANWDXXXU'], ['', 'P'], ['', 'PMS'], ['', 'PP'], ['', 'PPJY'], ['', 'PPSWRUGJQ'], ['', 'PWDEB'], ['', 'Q'], ['', 'QW'], ['', 'R'], ['', 'RLUB'], ['', 'RNMUSN'], ['', 'S'], ['', 'SD'], ['', 'SUBX

In [3]:
from collections import defaultdict

result = defaultdict(list)

for i, (key, _) in enumerate(data["rules"]):
    result[key].append(i)

second_refers = dict(result)
print(second_refers)

{'': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80], 'A': [81, 82, 83, 84, 85], 'ABG': [86], 'ADTB': [87], 'AIYOKM': [88], 'AJO': [89], 'AKPG': [90], 'AKUIUXNAHUR': [91], 'ANMN': [92], 'ANPFQFL': [93], 'AOTJANBOMJJYQJVUKJNH': [94], 'AQPXUA': [95], 'AQRTAQKUDRDOH': [96], 'ARJQCYALU': [97], 'ARTGXUWUVQUXXFKQ': [98], 'ASLRYIXXK': [99], 'AT': [100], 'AU': [101], 'AVBYOM': [102], 'AXS': [103], 'B': [104, 105, 106, 107], 'BBUOSBCV': [108], 'BDRPH': [109], 'BE': [110], 'BKBG': [111], 'BLM': [112], 'BLPJ': [113], 'BOYVMTAPD': [114], 'BRREHTSPMYLMA': [115], 'C': [116, 117, 118, 119, 120, 121], 'CAUNKEGP': [122], 'CBTQQJDYL': [123], 'CDQ': [124], 'CE': [125], 'CEQJ': [126], 'CFW': [127], 'CGTTX': [128], 'CJG': [129], 'CK': [130], 'C

In [4]:
import ahocorasick

automaton = ahocorasick.Automaton()
for idx, key in enumerate(list(second_refers.keys())[1:]):
    automaton.add_word(key, (idx, key))

automaton.make_automaton()

In [11]:
import json
import time

file_rule_path1 = "/content/test_parallel-2.jsonl"

def find_substrings(haystack, automaton):
    substrings = []
    for end_index, (insert_order, original_value) in automaton.iter(haystack):
        start_index = end_index - len(original_value) + 1
        substrings.append((original_value, start_index))
        assert haystack[start_index:start_index + len(original_value)] == original_value
    return substrings

time0 = time.time()
first_refers = dict()
p = 0
with open(file_rule_path1, "r") as f:
    for line in f:
        record = json.loads(line)
        traj = record.get("trajectory")
        if traj:  # Noneや空リストはスキップ
            for step in traj:
                p += 1
                haystack = step["state"]
                first_refers[haystack] = [('', i) for i in range(len(haystack) + 1)] + find_substrings(haystack, automaton)

print(f'the number of total data is {p}')
print(time.time()-time0)


the number of total data is 5176566
48.810760259628296


In [12]:
import json

# second_refersをJSONファイルとして保存
with open("second_refers.json", "w") as f:
    json.dump(second_refers, f)

print("second_refers.json を保存")

# first_refersをJSONファイルとして保存
# first_refersのkeyは文字列である必要があります
first_refers_str_keys = {str(k): v for k, v in first_refers.items()}
with open("first_refers.json", "w") as f:
    json.dump(first_refers_str_keys, f)

print("first_refers.json を保存")

second_refers.json を保存
first_refers.json を保存
